In [1]:
%%writefile app.py

import streamlit as st
import pandas as pd


st.set_page_config(
    page_title="Khwezi Mining",
    page_icon="💎",
    layout="wide"
)


st.markdown("""
<style>

/* MAIN APP */

.stApp {
    background-color: #F5F6F8;
}


/* HIDE STREAMLIT TOP BAR */

[data-testid="stHeader"] {
    display: none;
}

[data-testid="stToolbar"] {
    display: none;
}

[data-testid="stDecoration"] {
    display: none;
}

#MainMenu {
    visibility: hidden;
}

footer {
    visibility: hidden;
}


/* SIDEBAR */

[data-testid="stSidebar"] {
    background-color: #1B1B1B;
}

[data-testid="stSidebar"] * {
    color: white;
}


/* SIDEBAR NAVIGATION DROPDOWN */

[data-testid="stSidebar"] [data-baseweb="select"] > div {
    background-color: white;
    border-radius: 8px;
}

[data-testid="stSidebar"] [data-baseweb="select"] * {
    color: #1B1B1B !important;
}


/* MAIN HEADINGS */

h1 {
    color: #1B1B1B;
    font-weight: 700;
}

h2, h3 {
    color: #333333;
}


/* METRIC CARDS */

[data-testid="stMetric"] {
    background-color: white;
    padding: 18px;
    border-radius: 12px;
    border-left: 5px solid #D4AF37;
    box-shadow: 0px 2px 8px rgba(0,0,0,0.08);
}


/* BUTTONS */

.stButton > button {
    background-color: #D4AF37;
    color: black;
    border: none;
    border-radius: 8px;
    padding: 8px 20px;
    font-weight: bold;
}

.stButton > button:hover {
    background-color: #C29D2D;
    color: black;
}


/* DOWNLOAD BUTTONS */

.stDownloadButton > button {
    background-color: #1B1B1B;
    color: white;
    border-radius: 8px;
    border: none;
}

.stDownloadButton > button:hover {
    background-color: #D4AF37;
    color: black;
}


/* INPUT BOXES */

.stTextInput input {
    border-radius: 8px;
}

.stNumberInput input {
    border-radius: 8px;
}


/* DATA TABLES */

[data-testid="stDataFrame"] {
    background-color: white;
    border-radius: 10px;
}

</style>
""", unsafe_allow_html=True)


# LOAD CSV FILES

users = pd.read_csv("users.csv")
workers = pd.read_csv("workers.csv")
incidents = pd.read_csv("incidents.csv")
equipment = pd.read_csv("equipment.csv")


# PART A: ACCESS CONTROL

def login(username, password):

    user = users[
        (users["Username"] == username) &
        (users["Password"] == password)
    ]

    if len(user) == 1:
        return user["Role"].item()

    else:
        return "Invalid login"


permissions = {

    "Administrator": [
        "Dashboard",
        "Worker Safety",
        "Safety Incidents",
        "Equipment",
        "Maintenance",
        "Risk Assessment",
        "Alerts",
        "Data Analysis",
        "Reports",
        "Manage Users"
    ],

    "Safety Officer": [
        "Dashboard",
        "Worker Safety",
        "Safety Incidents",
        "Risk Assessment",
        "Alerts",
        "Data Analysis",
        "Reports"
    ],

    "Mining Engineer": [
        "Dashboard",
        "Worker Safety",
        "Safety Incidents",
        "Equipment",
        "Maintenance",
        "Risk Assessment",
        "Alerts",
        "Data Analysis",
        "Reports"
    ],

    "Maintenance Engineer": [
        "Dashboard",
        "Equipment",
        "Maintenance",
        "Alerts",
        "Data Analysis",
        "Reports"
    ],

    "Manager": [
        "Dashboard",
        "Worker Safety",
        "Safety Incidents",
        "Equipment",
        "Maintenance",
        "Risk Assessment",
        "Alerts",
        "Data Analysis",
        "Reports"
    ]
}


def get_permissions(role):

    if role in permissions:
        return permissions[role]

    else:
        return []


def show_manage_users():

    global users

    st.title("👤 Manage Users")

    st.subheader("Current Users")

    st.dataframe(
        users[["Username", "Role"]]
    )

    st.subheader("Add New User")

    new_username = st.text_input(
        "New Username"
    )

    new_password = st.text_input(
        "New Password",
        type="password"
    )

    new_role = st.selectbox(
        "New User Role",
        [
            "Administrator",
            "Safety Officer",
            "Mining Engineer",
            "Maintenance Engineer",
            "Manager"
        ],
        key="new_user_role"
    )

    if st.button("Add User"):

        if new_username == "" or new_password == "":

            st.error(
                "Please enter a username and password"
            )

        elif new_username in users["Username"].values:

            st.error(
                "Username already exists"
            )

        else:

            new_user = pd.DataFrame({
                "Username": [new_username],
                "Password": [new_password],
                "Role": [new_role]
            })

            users = pd.concat(
                [users, new_user],
                ignore_index=True
            )

            users.to_csv(
                "users.csv",
                index=False
            )

            st.success(
                "User added successfully"
            )

            st.rerun()


    st.subheader("Change User Role")

    selected_user = st.selectbox(
        "Select User",
        users["Username"],
        key="change_role_user"
    )

    selected_role = st.selectbox(
        "Select New Role",
        [
            "Administrator",
            "Safety Officer",
            "Mining Engineer",
            "Maintenance Engineer",
            "Manager"
        ],
        key="change_role"
    )

    if st.button("Change Role"):

        updated_roles = [""] * len(users)

        for i in range(len(users)):

            if users["Username"][i] == selected_user:

                updated_roles[i] = selected_role

            else:

                updated_roles[i] = users["Role"][i]

        users["Role"] = updated_roles

        users.to_csv(
            "users.csv",
            index=False
        )

        st.success(
            "User role changed successfully"
        )

        st.rerun()


    st.subheader("Delete User")

    delete_user = st.selectbox(
        "Select User to Delete",
        users["Username"],
        key="delete_user"
    )

    if st.button("Delete User"):

        users = users[
            users["Username"] != delete_user
        ].reset_index(drop=True)

        users.to_csv(
            "users.csv",
            index=False
        )

        st.success(
            "User deleted successfully"
        )

        st.rerun()


# PART B: WORKER HEALTH AND SAFETY MONITORING

def worker_risk(
    ppe,
    fatigue,
    near_misses,
    training,
    previous_incidents
):

    score = 0

    if ppe == "No":
        score = score + 2

    if fatigue == "High":
        score = score + 2

    elif fatigue == "Medium":
        score = score + 1

    if near_misses >= 2:
        score = score + 2

    if training == "Incomplete":
        score = score + 2

    if previous_incidents >= 2:
        score = score + 1

    if score >= 6:
        return "Critical"

    elif score >= 4:
        return "High"

    elif score >= 2:
        return "Medium"

    else:
        return "Low"


def safety_warning(
    ppe,
    training,
    fatigue,
    near_misses,
    previous_incidents
):

    warning = ""

    if ppe == "No":
        warning = warning + "PPE non-compliance, "

    if training == "Incomplete":
        warning = warning + "Training incomplete, "

    if fatigue == "High":
        warning = warning + "High fatigue, "

    if near_misses >= 2:
        warning = warning + "Multiple near misses, "

    if previous_incidents >= 2:
        warning = warning + "Multiple previous incidents, "

    if warning == "":
        return "No warning"

    return warning[:-2]


risk_levels = [""] * len(workers)

safety_warnings = [""] * len(workers)


for i in range(len(workers)):

    risk_levels[i] = worker_risk(
        workers["PPE_Compliance"][i],
        workers["Fatigue_Level"][i],
        workers["Near_Misses"][i],
        workers["Training_Status"][i],
        workers["Previous_Incidents"][i]
    )

    safety_warnings[i] = safety_warning(
        workers["PPE_Compliance"][i],
        workers["Training_Status"][i],
        workers["Fatigue_Level"][i],
        workers["Near_Misses"][i],
        workers["Previous_Incidents"][i]
    )


workers["Risk_Level"] = risk_levels

workers["Safety_Warning"] = safety_warnings


def show_worker_safety():

    st.title(
        "👷 Worker Health and Safety"
    )

    risk_filter = st.selectbox(
        "Filter by risk level",
        [
            "All",
            "Low",
            "Medium",
            "High",
            "Critical"
        ]
    )

    if risk_filter == "All":

        st.dataframe(
            workers
        )

    else:

        filtered_workers = workers[
            workers["Risk_Level"] == risk_filter
        ]

        st.dataframe(
            filtered_workers
        )

    st.subheader(
        "Workers Requiring Attention"
    )

    attention = workers[
        workers["Safety_Warning"] != "No warning"
    ]

    st.dataframe(
        attention
    )



        "What is the most common incident type?",

        "What is the most common cause?",

        "How many incidents are high or critical?",

        "What percentage of workers comply with PPE requirements?"
    ]

    equipment_questions = [

        "Which equipment has the highest downtime?",

        "Which equipment has the lowest availability?",

        "Which equipment has the highest temperature?",

        "Which equipment has the highest vibration?"
    ]

    if role == "Safety Officer":

        allowed_questions = safety_questions

    elif role == "Maintenance Engineer":

        allowed_questions = equipment_questions

    else:

        allowed_questions = questions

    question = st.selectbox(
        "Choose a mining question",
        allowed_questions
    )

    if st.button(
        "Answer Question"
    ):

        answer = answer_question(
            question
        )

        st.success(
            "Answer: " +
            str(answer)
        )

    if role != "Maintenance Engineer":

        st.subheader(
            "Safety Analysis"
        )

        st.write(
            "Incidents by Department"
        )

        st.bar_chart(
            incidents[
                "Department"
            ].value_counts()
        )

        st.write(
            "Incidents by Severity"
        )

        st.bar_chart(
            incidents[
                "Severity"
            ].value_counts()
        )

    if role != "Safety Officer":

        st.subheader(
            "Equipment Analysis"
        )

        st.write(
            "Equipment Condition"
        )

        st.bar_chart(
            equipment[
                "Condition_Status"
            ].value_counts()
        )


# REPORTS

def show_reports(role):

    st.title(
        "📄 Reports"
    )

    if role != "Maintenance Engineer":

        st.subheader(
            "Worker Safety Report"
        )

        st.dataframe(
            workers
        )

        worker_csv = workers.to_csv(
            index=False
        )

        st.download_button(
            "Download Worker Report",
            worker_csv,
            "worker_report.csv"
        )

        st.subheader(
            "Safety Incident Report"
        )

        st.dataframe(
            incidents
        )

        incident_csv = incidents.to_csv(
            index=False
        )

        st.download_button(
            "Download Incident Report",
            incident_csv,
            "incident_report.csv"
        )

    if role != "Safety Officer":

        st.subheader(
            "Equipment Report"
        )

        st.dataframe(
            equipment
        )

        equipment_csv = equipment.to_csv(
            index=False
        )

        st.download_button(
            "Download Equipment Report",
            equipment_csv,
            "equipment_report.csv"
        )


# LOGIN SESSION

if "logged_in" not in st.session_state:

    st.session_state.logged_in = False


if "role" not in st.session_state:

    st.session_state.role = None


# LOGIN PAGE

if st.session_state.logged_in == False or st.session_state.role == None:

    st.markdown(
        """
        <style>

        .block-container {
            padding-top: 0.3rem !important;
            padding-bottom: 1rem !important;
        }

        div[data-testid="stImage"] {
            margin-bottom: -1rem;
        }

        </style>
        """,
        unsafe_allow_html=True
    )


    left_logo, logo_column, right_logo = st.columns(
        [1.2, 1.5, 1.2]
    )


    with logo_column:

        st.image(
            "Khwezi Mining Diamond Emblem.png",
            width="stretch"
        )


    # MONITORING APPLICATION NAME

    st.markdown(
        """
        <div style="
            text-align: center;
            color: #1B1B1B;
            font-size: 20px;
            font-weight: 600;
            margin-top: -6px;
            margin-bottom: 20px;
        ">
            Mining Health, Safety & Equipment Monitoring Application
        </div>
        """,
        unsafe_allow_html=True
    )


    left, middle, right = st.columns(
        [1.4, 1.3, 1.4]
    )


    with middle:

        username = st.text_input(
            "Username"
        )

        password = st.text_input(
            "Password",
            type="password"
        )

        login_button = st.button(
            "Login",
            width="stretch"
        )


    if login_button:

        role = login(
            username,
            password
        )

        if role == "Invalid login":

            st.error(
                "Invalid username or password"
            )

        else:

            st.session_state.logged_in = True

            st.session_state.role = role

            st.rerun()


# MAIN APPLICATION

else:

    role = st.session_state.role


    # SIDEBAR BRAND

    st.sidebar.markdown(
        '<div style="display:flex; align-items:center; gap:12px; margin-bottom:20px;">'
        '<span style="color:#D4AF37; font-size:30px;">◆</span>'
        '<span style="color:white; font-size:24px; font-weight:700;">Khwezi Mining</span>'
        '</div>',
        unsafe_allow_html=True
    )


    st.sidebar.write(
        "Role:",
        role
    )


    pages = get_permissions(
        role
    )


    page = st.sidebar.selectbox(
        "Navigation",
        pages
    )


    if st.sidebar.button(
        "Logout"
    ):

        st.session_state.logged_in = False

        st.session_state.role = None

        st.rerun()


    # PART A

    if page == "Manage Users":

        show_manage_users()


    # PART B

    elif page == "Worker Safety":

        show_worker_safety()


    # PART C

    elif page == "Safety Incidents":

        show_safety_incidents()


    # PART D AND PART E

    elif page == "Equipment":

        show_equipment_database()

        show_equipment_condition()


    # PART F

    elif page == "Maintenance":

        show_maintenance()


    # PART G

    elif page == "Risk Assessment":

        show_risk_assessment()


    # PART H

    elif page == "Alerts":

        show_alerts(
            role
        )


    # PART I

    elif page == "Dashboard":

        show_dashboard()


    # PART J

    elif page == "Data Analysis":

        show_data_analysis(
            role
        )


    # REPORTS

    elif page == "Reports":

        show_reports(
            role
        )

Overwriting app.py


In [ ]:
!streamlit run app.py